# 11 · Merge MASLD adapters and push to Hugging Face
Merges a MASLD-Bench LoRA adapter into its Qwen3 base and pushes the merged
model (and the adapter repo) to your HF account.

Setup: add your HF token as a Colab secret named `HF_TOKEN` (key icon ->
Secrets), or paste it into the cell. NEVER commit it to a notebook.

Run on A100/high-RAM: the 8B merge in bf16 needs ~18 GB RAM.


In [10]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import subprocess, sys, importlib.metadata as md

# Same fix as bridge_worker.ipynb (proven 2026-10-30): peft's torchao dispatch
# raises ImportError with torchao<0.16 present; with torchao ABSENT it skips
# cleanly (plain LoRA needs no torchao). Uninstall; fall back to 0.7.0 if the
# image blocks it.
try:
    subprocess.run([sys.executable, "-m", "pip", "-q", "uninstall", "-y", "torchao"],
                   check=True)
    print("torchao: removed (peft dispatch inert — intended)")
except subprocess.CalledProcessError:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchao==0.7.0"],
                   check=True)
    print("torchao: downgraded to 0.7.0 (has LinearActivationQuantizedTensor)")

subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "transformers", "peft", "accelerate", "huggingface_hub"], check=True)

def _v(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return "absent"
print("peft", _v("peft"), "| transformers", _v("transformers"), "| torchao", _v("torchao"))



torchao: removed (peft dispatch inert — intended)
peft 0.21.2 | transformers 5.20.2 | torchao absent


In [18]:
# ---- config ----
import os
HF_REPO = "Subhadip-igtm/masld-bench-qwen3-8b-sft-v7"   # your-namespace/model-name
ADAPTER_DRIVE = "/content/drive/MyDrive/masld-llm/checkpoints/qwen3-8b-masld-sft-v7"
BASE_MODEL = "Qwen/Qwen3-8B"
# For the 8B: HF_REPO="Subhadip-igtm/masld-bench-qwen3-8b-sft-v7",
#   ADAPTER_DRIVE="/content/drive/MyDrive/masld-llm/checkpoints/qwen3-8b-masld-sft-v7",
#   BASE_MODEL="Qwen/Qwen3-8B"

# token: Colab secret first (key icon -> HF_TOKEN), then env, then cached login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
print("repo:", HF_REPO, "| token:", "set" if HF_TOKEN else "using cached login")



repo: Subhadip-igtm/masld-bench-qwen3-8b-sft-v7 | token: set


In [19]:
from huggingface_hub import HfApi, whoami
print("whoami:", whoami().get("name"))          # must print your username
print(HfApi().model_info("Qwen/Qwen3-1.7B").id) # must print Qwen/Qwen3-1.7B

whoami: Subhadip-igtm
Qwen/Qwen3-1.7B


In [20]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(BASE_MODEL, padding_side="left")
if tok.pad_token is None: tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, ADAPTER_DRIVE)
model = model.merge_and_unload()
print("merged")


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

merged


In [21]:
# smoke test before pushing
enc = tok("Answer yes or no:", return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=8, do_sample=False)
print("smoke:", tok.decode(out[0], skip_special_tokens=True))


smoke: Answer yes or no: Is the following a valid argument? If


In [22]:
from huggingface_hub import HfApi, login
if HF_TOKEN:
    login(token=HF_TOKEN)
# else: the runtime already has a cached token (huggingface-cli login)
model.push_to_hub(HF_REPO, private=True)
tok.push_to_hub(HF_REPO, private=True)
print("pushed merged model ->", HF_REPO)



Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...js0ysk6/model.safetensors:   0%|          | 24.0MB / 16.4GB            

Validating                    :           |   0%            

... (2 outputs left out)

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpnb2d77nz/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

pushed merged model -> Subhadip-igtm/masld-bench-qwen3-8b-sft-v7


In [23]:
# also push the bare adapter (small, for reproducibility)
from peft import PeftModel as _PM
adapter = _PM.from_pretrained(base, ADAPTER_DRIVE)
adapter.push_to_hub(HF_REPO + "-adapter", private=True)
print("pushed adapter repo ->", HF_REPO + "-adapter")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 57.0kB /  349MB            

Validating                    :           |   0%            

pushed adapter repo -> Subhadip-igtm/masld-bench-qwen3-8b-sft-v7-adapter


In [24]:
from huggingface_hub import HfApi
api = HfApi()
for repo, card in [
    ("Subhadip-igtm/masld-bench-qwen3-1.7b-sft-v6", "README_v6_1.7b.md"),
    ("Subhadip-igtm/masld-bench-qwen3-1.7b-sft-v6-adapter", "README_v6_1.7b.md"),
]:
    api.upload_file(
        path_or_fileobj=f"/content/drive/MyDrive/masld-llm/hf-release/{card}",
        path_in_repo="README.md",
        repo_id=repo,
        repo_type="model")
    print("card updated:", repo)

card updated: Subhadip-igtm/masld-bench-qwen3-1.7b-sft-v6
card updated: Subhadip-igtm/masld-bench-qwen3-1.7b-sft-v6-adapter


Then fill the model card on HF: training recipe (LLaMA-Factory LoRA/QLoRA,
v6/v7 mixture per the MANIFEST), the MB-89xx + MB-20xx leaderboard rows,
license (Apache-2.0 recommended, Qwen3-compatible), and the research-only
disclaimer (not for clinical use).
